# Lesson 3.2 — Invoking the Planner: Calling the Reference Layer

With a goal configuration, the Plan stage asks Module 7 for a timed, validated `reference(t)`. Module 9 only **calls, checks, and hands off** — no planning math.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
    to_configuration, plan_reference, fk_xy, P2, T2)
checks = []
world = GreenhouseWorld.demo_row(n=6, seed=1)
dets = model_perception(world, rng=np.random.default_rng(7))
_, target = understand(dets, world)
q_start = world.q.copy()
q_goal = to_configuration(target)


In [2]:
layer = plan_reference(q_start, q_goal, rng=np.random.default_rng(0))
print('validated:', layer['validated'], '| duration: %.3f s' % layer['duration'])
checks.append(layer['validated'] is True)


validated: True | duration: 1.130 s


### reference(t): a sampleable schedule of q_d, qd_d, qdd_d

In [3]:
q0, qd0, qdd0, info0 = layer['reference'](0.0)
qT, qdT, qddT, infoT = layer['reference'](layer['duration'])
print('reference(0)   q_d:', np.round(q0,3), '| phase:', round(info0['phase'],2))
print('reference(T)   q_d:', np.round(qT,3), '| phase:', round(infoT['phase'],2))
checks.append(np.allclose(q0, q_start, atol=1e-6))            # starts at q_start
checks.append(np.allclose(qT, q_goal, atol=1e-3))            # ends at q_goal


reference(0)   q_d: [0.4 0.8] | phase: 0.0
reference(T)   q_d: [-0.356  1.684] | phase: 1.0


### the reference endpoint reaches the target pose (FK check)

In [4]:
tool_T = fk_xy(P2, T2, qT)
print('FK(reference(T)):', np.round(tool_T,3), '| target:', np.round(target['xy'],3))
checks.append(np.linalg.norm(tool_T - target['xy']) < 1e-3)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


FK(reference(T)): [0.447 0.152] | target: [0.447 0.152]
4/4 checks passed.
All checks passed.
